[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/18_Breathing_Dynamics.ipynb)

# DiveLab

## Notebook 18 — Breathing Dynamics: Disturbance, Actuator and Fast Control Loop

**Guiding question:** How does breathing enter the dynamics of buoyancy control?

Notebook 17 introduced breathing in two roles:

$$
\boxed{\text{periodic disturbance}}
\qquad\text{and}\qquad
\boxed{\text{control input}}
$$

Now we make lung volume an explicit dynamical quantity and study how breathing interacts with depth, buoyancy and BCD control.

> **Important:** this is an educational control-system model. It is deliberately simplified and is not a prescription for real-world diving technique.

## Learning objectives

By the end of this notebook, you will be able to:

- distinguish tidal breathing from deliberate mean lung-volume adjustment;
- model lung-volume variation as a dynamical state;
- convert lung-volume change into buoyancy-force change;
- distinguish disturbance and control components of breathing;
- understand why breathing is a fast but limited actuator;
- compare breathing with slower BCD adjustment;
- model two control inputs acting on the same vertical plant;
- see how breathing can damp small depth deviations;
- understand actuator limits and saturation;
- connect breathing dynamics to multi-timescale and hierarchical control.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Lung volume affects buoyancy

Archimedes' principle gives:

$$
F_B=\rho gV_{\text{disp}}.
$$

A change in lung volume changes the diver's displaced volume approximately by the same amount:

$$
\Delta V_{\text{disp}}\approx\Delta V_L.
$$

Therefore:

$$
\boxed{
\Delta F_B\approx\rho g\Delta V_L
}
$$

A change of only a fraction of a litre can therefore create a noticeable buoyancy-force variation.

In [ ]:
rho = 1025.0
g = 9.80665

for litres in [0.1, 0.25, 0.5, 1.0]:
    dV = litres/1000
    dF = rho*g*dV
    print(f"{litres:4.2f} L -> {dF:5.2f} N")

# 2. Separate baseline and variation

Write lung volume as:

$$
V_L(t)
=
V_{L0}
+
\Delta V_L(t).
$$

The baseline $V_{L0}$ contributes to the operating-point buoyancy.

The variation:

$$
\Delta V_L(t)
$$

produces time-varying buoyancy.

# 3. Two components of breathing

For control analysis, it is useful to decompose:

$$
\boxed{
\Delta V_L(t)
=
V_T(t)+V_C(t)
}
$$

where:

- $V_T(t)$: cyclic tidal component;
- $V_C(t)$: deliberate shift of the breathing operating point.

The first can behave like a disturbance.

The second can behave like a control action.

# 4. Tidal breathing

A first approximation is sinusoidal:

$$
V_T(t)
=
A_T\sin(\omega_b t).
$$

This is not intended as a physiological model.

It simply captures the periodic character needed for dynamical analysis.

In [ ]:
A_T = 0.35/1000
f_b = 0.20
omega_b = 2*np.pi*f_b

t = np.linspace(0, 30, 1500)
V_tidal = A_T*np.sin(omega_b*t)

plt.plot(t, 1000*V_tidal)
plt.xlabel("Time [s]")
plt.ylabel("Lung-volume deviation [L]")
plt.title("Simplified tidal breathing")
plt.grid(True)
plt.show()

# 5. Tidal breathing produces cyclic buoyancy

Using:

$$
\Delta F_B=\rho gV_T,
$$

we obtain a periodic force.

In [ ]:
F_tidal = rho*g*V_tidal

plt.plot(t, F_tidal)
plt.xlabel("Time [s]")
plt.ylabel("Buoyancy-force variation [N]")
plt.title("Buoyancy variation caused by tidal breathing")
plt.grid(True)
plt.show()

If the diver made no deliberate correction, this periodic force could produce a small cyclic vertical motion.

In the control diagram, tidal breathing can therefore be represented as a disturbance entering the plant.

# 6. Deliberate breathing control

Now introduce:

$$
V_C(t).
$$

This represents a deliberate short-term shift in lung volume relative to the nominal breathing pattern.

Conceptually:

$$
V_C>0
$$

increases buoyancy, while:

$$
V_C<0
$$

reduces buoyancy.

Unlike the BCD, this control channel is naturally limited in magnitude and duration.

# 7. Breathing is not an instantaneous actuator

The diver cannot command lung volume to jump instantaneously.

A simple first-order actuator model is:

$$
\boxed{
\tau_L\dot V_C+V_C=V_{\text{cmd}}
}
$$

where:

- $V_{\text{cmd}}$ is the desired lung-volume shift;
- $V_C$ is the realized shift;
- $\tau_L$ is a response time constant.

In [ ]:
def simulate_lung_actuator(V_cmd_litres=0.4, tau_L=0.7, duration=8, dt=0.005):
    t = np.arange(0, duration+dt, dt)
    V = np.zeros_like(t)
    Vcmd = V_cmd_litres/1000

    for k in range(len(t)-1):
        dV = (Vcmd - V[k])/tau_L
        V[k+1] = V[k] + dV*dt

    return t, V

for tau_L in [0.3, 0.7, 1.5]:
    tt, VV = simulate_lung_actuator(tau_L=tau_L)
    plt.plot(tt, 1000*VV, label=f"tau_L={tau_L} s")

plt.xlabel("Time [s]")
plt.ylabel("Control lung-volume shift [L]")
plt.title("Breathing actuator dynamics")
plt.grid(True)
plt.legend()
plt.show()

# 8. Saturation

Breathing control has finite authority.

Represent this by:

$$
|V_{\text{cmd}}|
\le V_{\max}.
$$

This is analogous to actuator saturation in Notebook 16.

In [ ]:
Vmax = 0.5/1000

commands_L = np.linspace(-1.0, 1.0, 300)
commands = commands_L/1000
realized = np.clip(commands, -Vmax, Vmax)

plt.plot(commands_L, 1000*realized)
plt.xlabel("Requested shift [L]")
plt.ylabel("Allowed shift [L]")
plt.title("Simplified breathing-control saturation")
plt.grid(True)
plt.show()

# 9. Vertical plant with breathing input

Use depth $z$ positive downward and upward velocity $v$, so:

$$
\dot z=-v.
$$

A local vertical model is:

$$
m\dot v
=
F_{\text{instability}}
-
F_{\text{drag}}
+
\rho gV_C
+
\rho gV_T.
$$

We use a simplified linear form:

$$
\dot v
=
\alpha^2 z
-cv
+k_L(V_C+V_T).
$$

Here $k_L$ converts lung-volume variation into acceleration.

# 10. Open-loop tidal oscillation

Let's first apply tidal breathing without deliberate breathing control.

In [ ]:
def simulate_vertical(
    duration=40.0,
    dt=0.005,
    alpha=0.08,
    c=0.35,
    tidal_amp_L=0.35,
    tidal_freq=0.20,
    Kz=0.0,
    Kv=0.0,
    tau_L=0.7,
    Vmax_L=0.5,
    z0=0.0,
    v0=0.0
):
    t = np.arange(0, duration+dt, dt)
    z = np.zeros_like(t)
    v = np.zeros_like(t)
    Vc = np.zeros_like(t)
    Vtidal = np.zeros_like(t)
    Vcmd_hist = np.zeros_like(t)

    z[0] = z0
    v[0] = v0

    kL = rho*g/80.0  # illustrative 80 kg diver; acceleration per m^3

    for k in range(len(t)-1):
        Vtidal[k] = (tidal_amp_L/1000)*np.sin(2*np.pi*tidal_freq*t[k])

        # Fast breathing controller:
        # deeper than target -> positive lung-volume shift
        # upward velocity positive -> reduce additional buoyancy
        Vcmd = Kz*z[k] - Kv*v[k]
        Vcmd = np.clip(Vcmd, -Vmax_L/1000, Vmax_L/1000)
        Vcmd_hist[k] = Vcmd

        dVc = (Vcmd - Vc[k])/tau_L
        Vc[k+1] = Vc[k] + dVc*dt

        dv = alpha**2*z[k] - c*v[k] + kL*(Vc[k] + Vtidal[k])
        v[k+1] = v[k] + dv*dt
        z[k+1] = z[k] - v[k+1]*dt

    Vtidal[-1] = (tidal_amp_L/1000)*np.sin(2*np.pi*tidal_freq*t[-1])
    Vcmd_hist[-1] = Vcmd_hist[-2]

    return t, z, v, Vc, Vtidal, Vcmd_hist

In [ ]:
tt, z, v, Vc, Vtidal, Vcmd = simulate_vertical(alpha=0.0)

plt.plot(tt, z)
plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Vertical response to simplified tidal breathing")
plt.grid(True)
plt.show()

Even a zero-mean periodic force can produce cyclic depth and velocity variations.

Drag and inertia determine how much of the breathing oscillation appears as vertical motion.

# 11. Breathing as feedback control

Suppose the diver uses depth and vertical motion to modify mean lung volume:

$$
\boxed{
V_{\text{cmd}}
=
K_z z-K_vv
}
$$

Interpretation:

- $K_z z$: correct depth error;
- $-K_vv$: oppose vertical motion.

This resembles **PD control**.

The analogy is especially interesting:

$$
\text{depth error}
\longleftrightarrow P
$$

and

$$
\text{vertical velocity}
\longleftrightarrow D.
$$

The diver's breathing can therefore provide a small, fast stabilizing loop around the vertical dynamics.

In [ ]:
tt1, z1, v1, Vc1, Vt1, cmd1 = simulate_vertical(
    alpha=0.08,
    tidal_amp_L=0.0,
    z0=0.25,
    Kz=0.0,
    Kv=0.0
)

tt2, z2, v2, Vc2, Vt2, cmd2 = simulate_vertical(
    alpha=0.08,
    tidal_amp_L=0.0,
    z0=0.25,
    Kz=0.0015,
    Kv=0.0010
)

plt.plot(tt1, z1, label="No breathing feedback")
plt.plot(tt2, z2, label="Breathing feedback")
plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Fast breathing feedback")
plt.grid(True)
plt.legend()
plt.show()

# 12. But breathing has limited control authority

A large depth error cannot be corrected indefinitely by shifting lung volume.

The breathing channel saturates:

$$
|V_C|\le V_{\max}.
$$

This makes it suitable for **small and relatively fast corrections**, not arbitrary long-term buoyancy compensation.

In [ ]:
tt, z, v, Vc, Vt, cmd = simulate_vertical(
    alpha=0.08,
    tidal_amp_L=0.0,
    z0=1.5,
    Kz=0.003,
    Kv=0.001,
    Vmax_L=0.4
)

plt.plot(tt, 1000*Vc)
plt.axhline(0.4, linestyle="--")
plt.axhline(-0.4, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Breathing-control shift [L]")
plt.title("Breathing actuator reaches its limit")
plt.grid(True)
plt.show()

# 13. Why the BCD is still needed

Suppose a persistent buoyancy bias exists.

Using breathing alone would require the diver to maintain a nonzero lung-volume offset continuously.

A slower BCD adjustment can instead change the baseline buoyancy.

This suggests a natural division of labor:

$$
\boxed{
\text{breathing: transient correction}
}
$$

$$
\boxed{
\text{BCD: persistent baseline correction}
}
$$

# 14. Two actuators

Let:

$$
u_L=V_C
$$

be the breathing control input, and:

$$
u_B=V_{\text{BCD}}
$$

be BCD gas-volume adjustment.

Then:

$$
m\dot v
=
F_0
+
\rho g u_L
+
\rho g u_B
-
F_D.
$$

The plant is now a **multi-input system**.

In state-space form:

$$
\dot x
=
Ax
+
B_Lu_L
+
B_Bu_B
+
Ed.
$$

This is a natural extension of the state-space models introduced earlier in DiveLab.

# 15. Fast and slow loops

We can now make the architecture more explicit:

```text
                    FAST CONTROL
depth + velocity ------> breathing ------+
                                         |
                                         v
                                      DIVER
                                         ^
                                         |
depth trend -----------> BCD ------------+
                    SLOW CONTROL
```

The two loops act on the same plant but operate at different characteristic time scales.

# 16. Hierarchical interpretation

A higher-level strategy could be:

1. use breathing for small transient corrections;
2. observe whether a persistent breathing offset is needed;
3. if the offset persists, change baseline BCD buoyancy;
4. return breathing toward its nominal operating region.

This resembles **hierarchical control**.

In engineering language, the fast actuator handles short-term regulation while the slow actuator handles low-frequency or steady-state demand.

This avoids using a limited fast actuator to solve a persistent problem.

# 17. A simple two-timescale controller

We can model:

$$
V_{\text{cmd}}
=
K_z z-K_vv
$$

for fast breathing control.

For the BCD, use a much slower integral-like adaptation:

$$
\dot V_B
=
K_B z.
$$

A persistent depth error gradually changes baseline buoyancy.

In [ ]:
def simulate_two_loop(
    duration=80,
    dt=0.01,
    z0=0.5,
    alpha=0.08,
    c=0.35,
    Kz=0.0015,
    Kv=0.0010,
    KB=2e-6,
    tau_L=0.7,
    Vmax_L=0.4,
    persistent_bias=-0.025
):
    t = np.arange(0, duration+dt, dt)

    z = np.zeros_like(t)
    v = np.zeros_like(t)
    VL = np.zeros_like(t)
    VB = np.zeros_like(t)

    z[0] = z0
    kL = rho*g/80.0

    for k in range(len(t)-1):
        VL_cmd = Kz*z[k] - Kv*v[k]
        VL_cmd = np.clip(VL_cmd, -Vmax_L/1000, Vmax_L/1000)

        VL[k+1] = VL[k] + ((VL_cmd - VL[k])/tau_L)*dt

        # slow baseline adaptation
        VB[k+1] = VB[k] + KB*z[k]*dt

        dv = (
            alpha**2*z[k]
            - c*v[k]
            + kL*(VL[k] + VB[k])
            + persistent_bias
        )

        v[k+1] = v[k] + dv*dt
        z[k+1] = z[k] - v[k+1]*dt

    return t, z, v, VL, VB

In [ ]:
tt, z, v, VL, VB = simulate_two_loop()

plt.plot(tt, z)
plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Fast breathing loop + slow baseline adaptation")
plt.grid(True)
plt.show()

In [ ]:
plt.plot(tt, 1000*VL, label="Breathing control")
plt.plot(tt, 1000*VB, label="Slow BCD baseline")
plt.xlabel("Time [s]")
plt.ylabel("Equivalent volume adjustment [L]")
plt.title("Two control time scales")
plt.grid(True)
plt.legend()
plt.show()

Notice the intended division:

- breathing responds quickly;
- the slow baseline adjustment gradually carries the persistent load.

The breathing control can then return closer to its nominal operating region.

# 18. Connection to integral control

The slow rule:

$$
\dot V_B=K_Bz
$$

means:

$$
V_B(t)
=
K_B\int z(t)\,dt.
$$

So the slow BCD adaptation behaves mathematically like an **integral controller**.

The combined architecture resembles:

- breathing: fast P/PD-like action;
- BCD: slow integral-like baseline adaptation.

This gives a physically intuitive interpretation of PID-like control:

$$
\boxed{
P/D \rightarrow \text{rapid transient correction}
}
$$

$$
\boxed{
I \rightarrow \text{slow persistent buoyancy adjustment}
}
$$

The mapping is conceptual rather than literal, but it is useful for understanding control architecture.

# 19. Breathing and frequency response

Notebook 17 showed that breathing has a characteristic frequency.

Now we can see that breathing contains at least two frequency components:

### Tidal component

Periodic oscillation around the operating point.

### Control component

Slower or faster deliberate shifts of the operating point.

So breathing itself contains both **oscillatory dynamics** and **control dynamics**.

# 20. Breathing and observability

The diver does not directly measure lung volume with a sensor.

Instead, its effect is inferred through:

- bodily sensation;
- depth trend;
- vertical velocity;
- visual reference;
- pressure/depth instruments.

So breathing control also connects to the observability and state-estimation ideas from Notebooks 07–09.

# 21. Pressure dependence — an important distinction

Gas in a flexible BCD changes volume substantially with ambient pressure.

The lungs are different during normal scuba breathing because the regulator supplies gas approximately at ambient pressure and the diver actively ventilates.

Therefore we should **not simply treat the lungs as a sealed Boyle-law balloon** during normal breathing.

Pressure, respiration and physiology interact, and a detailed respiratory model is beyond this notebook.

For our control model, we treat voluntary lung-volume variation locally around an operating point.

# 22. Limits of the model

Our model ignores many effects, including:

- realistic respiratory physiology;
- airway dynamics;
- gas density and work of breathing;
- posture and trim;
- changing drag geometry;
- nonlinear lung compliance;
- coupling between breathing and metabolic demand;
- detailed regulator dynamics.

That is intentional.

The objective is to isolate the **control-system structure**.

# 23. Systems insight

The interesting result is not merely that breathing changes buoyancy.

It is that the diver has **multiple actuators with different dynamics and limits**.

That immediately raises advanced control questions:

- How should control effort be allocated?
- Which actuator should handle which frequency range?
- What happens when one actuator saturates?
- How should the slow loop estimate persistent bias?
- How does delay affect coordination?

These are general control-engineering questions.

# Exercises

### 1. Buoyancy per litre

Compute the buoyancy-force change produced by:

$$
0.2,\ 0.4,\ 0.6\ \text{L}
$$

of lung-volume variation in freshwater and seawater.

In [ ]:
# Your code here

### 2. Lung actuator time constant

Vary:

$$
\tau_L.
$$

How does it affect the response to a commanded lung-volume shift?

In [ ]:
# Your code here

### 3. Tidal frequency

Change breathing frequency and amplitude.

Observe the resulting depth oscillation.

How do drag and inertia affect the result?

In [ ]:
# Your code here

### 4. Fast breathing feedback

Experiment with:

$$
K_z,\qquad K_v.
$$

Find values that damp a small initial depth deviation without excessive lung-volume demand.

In [ ]:
# Your code here

### 5. Slow BCD adaptation

Change:

$$
K_B.
$$

What happens if the slow loop is too slow?

What happens if it becomes too aggressive?

In [ ]:
# Your code here

# Challenge — control allocation

Design a simple rule that divides required buoyancy correction between:

$$
u_L
$$

and:

$$
u_B.
$$

Try to satisfy:

- breathing handles rapid small corrections;
- BCD handles persistent correction;
- breathing remains inside a comfortable modeled range;
- control does not oscillate.

Plot depth, breathing control and BCD control.

This is a first introduction to **control allocation**.

In [ ]:
# Your code here

# Summary

Breathing is more than a periodic disturbance.

We modeled:

$$
V_L
=
V_{L0}
+
V_T
+
V_C.
$$

Here:

- $V_T$ is tidal breathing;
- $V_C$ is deliberate control variation.

Lung-volume change produces buoyancy change:

$$
\Delta F_B
\approx
\rho g\Delta V_L.
$$

We introduced breathing actuator dynamics:

$$
\tau_L\dot V_C+V_C=V_{\text{cmd}}.
$$

And a fast feedback law:

$$
V_{\text{cmd}}
=
K_z z-K_vv.
$$

Breathing is:

$$
\boxed{\text{fast but limited}}
$$

while BCD adjustment is naturally interpreted as:

$$
\boxed{\text{slower baseline control}}.
$$

Together they form a multi-input, multi-timescale control architecture.

### Core systems insight

$$
\boxed{
\text{small transient error}
\rightarrow
\text{fast breathing correction}
}
$$

$$
\boxed{
\text{persistent bias}
\rightarrow
\text{slow baseline adjustment}
}
$$

### Next — Notebook 19

According to our roadmap, the next advanced topic is **Gas Consumption**.

We can connect:

$$
\text{depth}
\rightarrow
\text{ambient pressure}
\rightarrow
\text{gas consumption rate}
$$

and build a dynamic gas-state model:

$$
\dot G=-q(P,\text{breathing demand}).
$$

That will connect physics, breathing and resource management in a single state-space model.